In [ ]:

import os
import re
from pathlib import Path
import pandas as pd


def find_repo_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "llm_pipeline").exists() and (candidate / "CV_analysis").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing llm_pipeline and CV_analysis.")


REPO_ROOT = find_repo_root()
CV_ROOT = REPO_ROOT / "CV_analysis"

# === Paths ===
RUNS_CSV = REPO_ROOT / "llm_pipeline/outputs_export/cv_runs_fixed.csv"
STEMS_CSV = CV_ROOT / "lexica/tum_gender_stems.csv"
WORDS_CSV = CV_ROOT / "lexica/tum_gender_words.csv"
NAMES_CSV = CV_ROOT / "data/CV_names_1985.csv"
LEXICON_CERTAINTY = CV_ROOT / "lexica/certain.csv"
LEXICON_TENTATIVE = CV_ROOT / "lexica/tentative.csv"
STEMS_TENTATIVE = CV_ROOT / "lexica/tentative_words_umlaut_only.csv"
STEMS_CERTAINTY = CV_ROOT / "lexica/certain_words_umlaut_only.csv"
WORD_COUNT_DIR = CV_ROOT / "data/word_counts"
WORD_COUNT_DIR.mkdir(parents=True, exist_ok=True)

df_all = pd.read_csv(RUNS_CSV)
print("Loaded", RUNS_CSV, df_all.shape)
print(df_all["provider"].value_counts().to_string())


In [ ]:
import warnings
import ast

def extract_all_text_without_personal(payload):
    """
    Entfernt kompletten Block '01_persoenliche_daten' aus dem JSON
    und extrahiert sonst alle Strings rekursiv.
    """
    if isinstance(payload, dict) and "01_persoenliche_daten" in payload:
        payload = {k: v for k, v in payload.items() if k != "01_persoenliche_daten"}

    texts = []

    def rec(x):
        if isinstance(x, dict):
            for v in x.values():
                rec(v)
        elif isinstance(x, list):
            for v in x:
                rec(v)
        elif isinstance(x, str):
            t = x.strip()
            if t:
                texts.append(t)

    rec(payload)
    return "\n".join(texts)


def get_text_from_row(row):
    """
    Versucht zuerst, response_json als Python-Objekt zu parsen und
    via extract_all_text_without_personal Text zu extrahieren.
    Falls das fehlschlägt, fällt auf response_text (oder raw_json) zurück.
    """
    raw_json = row.get("response_json", "")
    fallback_text = row.get("response_text", "") or raw_json or ""

    # 1) Versuch: strukturiertes JSON/Python-Objekt
    if isinstance(raw_json, str) and raw_json.strip():
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", SyntaxWarning)
                payload = ast.literal_eval(raw_json)
            return extract_all_text_without_personal(payload)
        except Exception:
            # wenn das nicht geht, unten Fallback
            pass

    # 2) Fallback: einfach roher Text
    return str(fallback_text)

def extract_json_snippet(text: str) -> str | None:
    """
    Versucht, aus einem freien Text den JSON-ähnlichen Teil zu extrahieren.
    Nimmt alles zwischen dem ersten '{' und dem letzten '}'.
    Entfernt ggf. ```-Code-Fences.
    """
    if not isinstance(text, str):
        return None

    # Code-Fences grob rausnehmen
    cleaned = text.strip()
    for fence in ("```json", "```JSON", "```"):
        if cleaned.startswith(fence):
            cleaned = cleaned[len(fence):].lstrip()
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3].rstrip()

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start == -1 or end == -1 or end <= start:
        return None

    snippet = cleaned[start:end+1].strip()
    return snippet if snippet else None


def fill_missing_response_json(df: pd.DataFrame) -> pd.DataFrame:
    """
    Füllt fehlende response_json-Einträge, indem für diese Zeilen
    aus response_text ein JSON-ähnlicher Block extrahiert wird.
    Gibt eine kopierte, modifizierte Version von df zurück.
    """
    df = df.copy()

    mask = df["response_json"].isna() & df["response_text"].notna()
    idx_missing = df[mask].index

    for idx in idx_missing:
        txt = df.at[idx, "response_text"]
        snippet = extract_json_snippet(txt)
        if snippet is not None:
            df.at[idx, "response_json"] = snippet

    return df


In [ ]:

# Define provider DataFrames.
# Qwen provider labels are local-provider names from the pipeline export.
df_openai = df_all[df_all["provider"] == "openai"].copy()
df_gemini = fill_missing_response_json(df_all[df_all["provider"] == "google-gemini"].copy())
df_qwen4B = df_all[df_all["provider"].astype(str).str.contains("qwen3-4b", case=False, na=False)].copy()
df_qwen8B = df_all[df_all["provider"].astype(str).str.contains("qwen3-8b", case=False, na=False)].copy()

MODEL_RUNS = {
    "openai": df_openai,
    "gemini": df_gemini,
    "qwen4B": df_qwen4B,
    "qwen8B": df_qwen8B,
}

OUT_PATHS = {
    "openai": WORD_COUNT_DIR / "openai_agentic_communal_tentative_certain.csv",
    "gemini": WORD_COUNT_DIR / "gemini_agentic_communal_tentative_certain.csv",
    "qwen4B": WORD_COUNT_DIR / "qwen4B_agentic_communal_tentative_certain.csv",
    "qwen8B": WORD_COUNT_DIR / "qwen8B_agentic_communal_tentative_certain.csv",
}

for model_name, model_df in MODEL_RUNS.items():
    print(f"{model_name}: {len(model_df)} rows")


In [20]:
os.getcwd()

'/Users/charlotteleininger/Master/Consulting/CV_analysis/code/Lexica_based'

In [44]:
TOKEN_RE = re.compile(r"[a-zäöüß\-]+", re.IGNORECASE)


def tokenize(s):
    if not isinstance(s, str):
        return []
    return TOKEN_RE.findall(s.lower())


def load_tum_stems(stems_csv):
    df = pd.read_csv(stems_csv)
    df["stem"] = df["stem"].astype(str).str.strip().str.lower()
    agentic = df[df["category"] == "agentic"]["stem"].tolist()
    communal = df[df["category"] == "communal"]["stem"].tolist()
    return agentic, communal


def load_tum_words(words_csv):
    if not os.path.exists(words_csv):
        return set(), set()
    df = pd.read_csv(words_csv)
    df["word"] = df["word"].astype(str).str.strip().str.lower()
    return (
        set(df[df["category"] == "agentic"]["word"].tolist()),
        set(df[df["category"] == "communal"]["word"].tolist())
    )


def load_stems_csv(path):
    df = pd.read_csv(path)
    col = df.columns[0]
    stems = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )
    return [s for s in stems if s]



def load_lexicon(path):
    """
    Einfache Loader-Funktion für certainty/tentative Lexika.
    Nimmt erste Spalte, lowercased, entfernt leere Einträge.
    Versucht optional, eine Header-Zeile wie 'word' zu droppen.
    """
    df = pd.read_csv(path, header=None)
    words = (
        df.iloc[:, 0]
        .astype(str)
        .str.strip()
        .str.lower()
    )
    if len(words) > 0 and words.iloc[0] in {"word", "wort", "term"}:
        words = words.iloc[1:]
    return set(w for w in words if w)


def build_patterns(stems):
    pats = []
    for s in stems:
        if isinstance(s, str) and len(s) > 1:
            pats.append(re.compile(rf"^{re.escape(s)}"))
    return pats


def count_stems(tokens, patterns):
    c = 0
    for t in tokens:
        for p in patterns:
            if p.match(t):
                c += 1
                break
    return c


def extract_all_text_without_personal(payload):
    """
    Entfernt kompletten Block '01_persoenliche_daten' aus dem JSON
    und extrahiert sonst alle Strings rekursiv.
    """
    if isinstance(payload, dict) and "01_persoenliche_daten" in payload:
        payload = {k: v for k, v in payload.items() if k != "01_persoenliche_daten"}

    texts = []

    def rec(x):
        if isinstance(x, dict):
            for v in x.values():
                rec(v)
        elif isinstance(x, list):
            for v in x:
                rec(v)
        elif isinstance(x, str):
            t = x.strip()
            if t:
                texts.append(t)

    rec(payload)
    return "\n".join(texts)


In [22]:
### clean certainty and tentative lexicons

def ascii_to_umlaut(word: str) -> str:
    """
    Wandelt ae/oe/ue in ä/ö/ü um.
    Trailing "*" (Stamm-Markierung) bleibt hängen.
    """
    star = ""
    if word.endswith("*"):
        word_core = word[:-1]
        star = "*"
    else:
        word_core = word

    # einfache Heuristik für deutsche ae/oe/ue-Umlaute
    word_core = (
        word_core
        .replace("ae", "ä")
        .replace("oe", "ö")
        .replace("ue", "ü")
        .replace("Ae", "Ä")
        .replace("Oe", "Ö")
        .replace("Ue", "Ü")
    )
    return word_core + star


def prefer_umlauts(words):
    """
    Nimmt eine Liste von Wörtern/Stämmen und entfernt die ae/oe/ue-Varianten,
    wenn die entsprechende Umlaut-Form in der Liste vorhanden ist.
    Reihenfolge der Liste bleibt ansonsten erhalten.
    """
    # Original-Set, um schnell zu prüfen, ob eine Umlaut-Variante existiert
    word_set = set(words)

    result = []
    seen = set()

    for w in words:
        if not w or w in seen:
            continue

        # nur interessant, wenn „ae/oe/ue“ überhaupt vorkommt
        if any(x in w for x in ("ae", "oe", "ue", "Ae", "Oe", "Ue")):
            umlaut_candidate = ascii_to_umlaut(w)
            # wenn die Umlaut-Variante in der Liste ist -> ASCII-Variante skippen
            if umlaut_candidate in word_set:
                # ASCII-Variante nicht behalten
                seen.add(w)
                continue

        result.append(w)
        seen.add(w)

    return result


In [ ]:

def load_wordlist(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


import csv

def save_wordlist_csv(words, path):
    """
    Speichert eine Liste von Wörtern als CSV mit einer Spalte 'word'.
    """
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["word"])
        for w in words:
            writer.writerow([w])


tentative_raw = load_wordlist(LEXICON_TENTATIVE)
tentative_clean = prefer_umlauts(tentative_raw)
# save_wordlist_csv(tentative_clean, CV_ROOT / "lexica/tentative_words_umlaut_only.csv")

certainty_raw = load_wordlist(LEXICON_CERTAINTY)
certainty_clean = prefer_umlauts(certainty_raw)
# save_wordlist_csv(certainty_clean, CV_ROOT / "lexica/certain_words_umlaut_only.csv")


In [48]:
# ===================================================================
# UNIVERSAL PROCESSOR
# ===================================================================
def process_df(
    df,
    df_names,
    pat_agentic,
    pat_communal,
    pat_certainty,
    pat_tentative,
):
    df = df.copy()
    df["first_name_lower"] = df["first_name"].str.lower().str.strip()
    df["last_name_lower"] = df["last_name"].str.lower().str.strip()

    df = df.merge(
        df_names.rename(columns={
            "first_name": "first_name_lower",
            "last_name": "last_name_lower"
        }),
        on=["first_name_lower", "last_name_lower"],
        how="left"
    )

    results = []

    for _, row in df.iterrows():
        text = get_text_from_row(row)
        toks = tokenize(text)
        n = len(toks)

        if n == 0:
            agentic_cnt = 0
            communal_cnt = 0
            cert_cnt = 0
            tent_cnt = 0
        else:
            agentic_cnt = count_stems(toks, pat_agentic)
            communal_cnt = count_stems(toks, pat_communal)
            cert_cnt = count_stems(toks, pat_certainty)
            tent_cnt = count_stems(toks, pat_tentative)

        results.append({
            "profile_id": row["profile_id"],
            "first_name": row["first_name"],
            "last_name": row["last_name"],
            "gender": row.get("gender"),
            "ethnicity": row.get("ethnicity"),
            "name_ID": row.get("name_ID"),
            "tokens": n,
            "agentic_count": agentic_cnt,
            "communal_count": communal_cnt,
            "certainty_count": cert_cnt,
            "tentative_count": tent_cnt,
        })

    return pd.DataFrame(results)


In [ ]:

# ===================================================================
# MAIN
# ===================================================================
def build_lexical_patterns():
    # TUM-Stems für agentic/communal
    agentic_stems, communal_stems = load_tum_stems(STEMS_CSV)

    # Stems für certainty / tentative (Umlaut-Versionen)
    certainty_stems = load_stems_csv(STEMS_CERTAINTY)
    tentative_stems = load_stems_csv(STEMS_TENTATIVE)

    return {
        "agentic": build_patterns(agentic_stems),
        "communal": build_patterns(communal_stems),
        "certainty": build_patterns(certainty_stems),
        "tentative": build_patterns(tentative_stems),
    }


def run_lexical_counts(model_runs=MODEL_RUNS, out_paths=OUT_PATHS):
    df_names = pd.read_csv(NAMES_CSV)
    df_names["first_name"] = df_names["first_name"].str.strip().str.lower()
    df_names["last_name"] = df_names["last_name"].str.strip().str.lower()

    patterns = build_lexical_patterns()
    outputs = {}

    for model_name, model_df in model_runs.items():
        df_counts = process_df(
            model_df,
            df_names,
            patterns["agentic"],
            patterns["communal"],
            patterns["certainty"],
            patterns["tentative"],
        )
        out_path = out_paths[model_name]
        df_counts.to_csv(out_path, index=False)
        outputs[model_name] = df_counts
        print(f"{model_name}: saved {len(df_counts)} rows to {out_path}")

    return outputs


lexical_outputs = run_lexical_counts()
